<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [16]</a>'.</span>

# Tennessee Eastman Process — Adaption

Adaptiert das in *tep_model* trainierte MLP an den Fehlerfallstrom (IDV 29); Module `drift_adaptation.py` und `drift_detection.py`. Verglichen werden die Baseline ohne Adaption und drei Strategien nach Gama et al. (2014, Abschn. 3.3.2), jeweils ohne und mit Recurrence-Selektor (Masterarbeit Abschn. 5.5.3, Rückholung vergangener Punkte mit ähnlichem Fehler):

- *passiv* (`blind`): periodisches Nachtraining ohne Fehlerauswertung,
- *aktiv* (`informed`): Nachtraining nach einer Detektion, mit Nachführphase und Refraktärzeit,
- *kombiniert* (`combined`): aktiv, mit periodischem Nachtraining als Rückfallebene.

Als Trigger werden fünf Detektoren gescreent: die vier statistischen Verfahren und der Schwellwert-Detektor `RMSE`, die einfachste Form der Überwachung eines Performance-Indikators (Gama Abschn. 3.2).

Aufbau und Kaskade sind deckungsgleich zu *syn_adaptation*. TEP-spezifisch sind die Datenquelle (`data_scored_error`, Zielgröße `cost`), die Abtastrate (`SAMPLES_PER_H`) und das Betriebspunktsignal `mode`, aus dem die stationäre Phasendauer $T_\mathrm{stat}$ geschätzt wird. Das Driftsignal wird als Spalte `driftf` gelesen (wie in *tep_detection*).

**Ablauf**

1. Einstufiges Tuning aller Strategien auf einem gröber abgetasteten Strom, mit gleichem Suchraum-Code, Budget und Sampler.
2. Detektor-Screening und Gegenprüfung auf voller Auflösung über mehrere Seeds.
3. Selektorparameter je Strategie bei festen Strategieparametern.
4. Auswertung über `N_SEED_RUNS` Seeds, Abbildungen, Ergebnisdatei und LaTeX-Makros.

**Status.** Fortschritt und Restzeit jeder Stufe stehen im Log, in `_temp/status/<notebook>.json` (Übersicht mit `python3 src/utils/run_status.py`) und, falls Zugangsdaten hinterlegt sind, im Telegram-Chat.

`FORCE_RECOMPUTE` rechnet alle Cache-Stufen neu, `IS_FINAL` schreibt zusätzlich den stabilen, in die Arbeit eingebundenen Stand von Abbildungen und Ergebnissen.

In [1]:
FORCE_RECOMPUTE = False
IS_FINAL = True
RUN_PHASE = "full"

In [2]:
# Parameters
RUN_PHASE = "tune"


## Konstanten

`RUN_ID` und `ERROR_VARIANT` wählen Modell und Fehlerfall (`None`: jeweils der neueste). Adaptions- und Tuningraster folgen *syn_adaptation* (dort 600 s und 3600 s auf einem 60-s-Rohstrom). Der TEP-Rohstrom ist mit 180 s bereits gröber: `STEP_S = 1800` s hält den Adaptionsstrom bei rund 35 000 Punkten, `STEP_TUNE_S = 10800` s das Tuningraster bei rund 5 800, mit demselben Verhältnis 6 wie im synthetischen Fall.

Alle Strategien erhalten im Tuning dasselbe Budget `N_TRIALS["final"]`, das Detektor-Screening ein reduziertes. Die Selektorparameter haben einen eigenen, kleineren Suchraum mit `REC_TRIALS` Versuchen. `COOLDOWN_X` ist die Refraktärzeit als Vielfaches von $T_\mathrm{stat}$ für die Schätzung; getunt wird sie als `cooldown_x` in [0, 60]. `INT_UPPER` hebt Obergrenzen einzelner Ganzzahl-Parameter des Suchraums an, ohne dessen Code und damit `SPACE_ID` zu ändern. Er geht nur in den Cache-Schlüssel ein, wenn er nicht leer ist, sodass Läufe ohne Abweichung ihre Schlüssel behalten. Die Grenzen selbst stehen in `da.search_bounds`; `INT_UPPER` ist derzeit leer. `TOP_K` ist die Zahl der Kandidaten je Strategie in der Gegenprüfung. `REJECT` verwirft ein Nachtraining, wenn der MSE dadurch nicht sinkt. `SCREENING_SEEDS` sind der Haupt-Seed und vier daraus reproduzierbar gezogene, über sie entscheiden Screening und Gegenprüfung. `TIE_ALPHA`, `TIE_MAX_REL` und `TIE_ORDER` bilden die Gleichstandsregel (siehe Gegenprüfung des Screenings), `PROBE_FACTORS` die Kandidaten der Randsonde. `RUN_PHASE` (papermill-Parameter) wählt die Phase: `probe` endet nach der Randsonde, `tune` vor den Seed-Läufen, `full` rechnet alles.

In [3]:
RUN_ID = "8165f7ca"
ERROR_VARIANT = {"idv": 29, "amp": 0.3}
NB_NAME = "tep_adaptation"

SEED = 42

SAMPLES_PER_H = 20
SAMPLE_S = 3600 // SAMPLES_PER_H

STEP_S = 1800
STEP_TUNE_S = 10800
DETECTORS = ("KSWIN", "ADWIN", "DDM", "EDDM", "RMSE")
N_TRIALS = {"screening": 40, "final": 60}
REC_TRIALS = 30
TOP_K = 3
INT_UPPER = {}
TUNE_WEIGHTS = (1.0, 0.1)
COOLDOWN_X = 30
REJECT = True
TIE_ORDER = ("RMSE", "DDM", "EDDM", "ADWIN", "KSWIN")
TIE_ALPHA = 0.05
TIE_MAX_REL = 0.05
PROBE_FACTORS = (1.5, 2.0, 4.0)

N_SEED_RUNS = 50

import numpy as np
SCREENING_SEEDS = (SEED, *np.random.default_rng(SEED).integers(1, 2**10 - 1, size=4).tolist())

PLOT_FRAC = 0.05
PLOT_YLIM = (-1e2, 1e2)
PLOT_EPOCH = "1970-01-01"
PLOT_MONTHS = 6

In [4]:
import os
from pathlib import Path
from src.utils.run_status import RunStatus

work_dir = os.getcwd()
DEFAULT_BASE_DIR = os.path.normpath(os.path.join(work_dir, "..", ".."))

base_dir = Path(os.environ.get("BASE_DIR", DEFAULT_BASE_DIR)).resolve()
data_dir = base_dir / "data" / "tep"
model_dir = base_dir / "models"
plot_dir = base_dir / "plots"
results_dir = base_dir / "results"

for _d in (data_dir, model_dir, plot_dir, results_dir):
    os.makedirs(_d, exist_ok=True)

status = RunStatus(NB_NAME, base_dir / "_temp" / "status")

## Daten und Basismodell laden

`assert_current` bricht ab, wenn ein Modul im Kernel veraltet ist. `base_cfg` (Modell und Fehlerfall) ist der gemeinsame Kopf aller Cache-Schlüssel dieses Notebooks.

In [5]:
import numpy as np
import pandas as pd
from src.utils import run_registry as rr
rr.assert_current()

ledger = rr.run_ledger(work_dir)
data_store = rr.tep_data_store(data_dir)
model_store = rr.tep_model_store(model_dir)
plot_store = rr.tep_plot_store(plot_dir, ledger=ledger)
res_store = rr.tep_results_store(results_dir, ledger=ledger)

rng = np.random.default_rng(SEED)

_mp = model_store.latest("model")
if _mp is None:
    raise FileNotFoundError("Kein model-Artefakt. Bitte tep_model.ipynb ausfuehren.")
model_id = model_store.parse_id(_mp, "model") if RUN_ID is None else RUN_ID
_mc = model_store.load("model", run_id=model_id, rename=True)
base_model, feature_cols = _mc["model"], _mc["feature_cols"]

test_scored = data_store.load("data_scored_production", run_id=model_id, rename=True)
model_cfg = rr.RunConfig.from_document(test_scored.attrs.get("config", test_scored.attrs))

error_scored, scored_error_cfg = rr.load_section(
    data_store, "data_scored_error", model_cfg, "error", ERROR_VARIANT)
error_variant = scored_error_cfg.sections["error"]
ledger.bind("adaptation", parents={"model": scored_error_cfg.id})

base_cfg = model_cfg.compose(error=error_variant)
print(f"model_id = {model_id} | error_run_id = {scored_error_cfg.id} "
      f"(IDV {error_variant['idv']}, amp {error_variant['amp']})")

I0000 00:00:1793633484.331429       7 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1793633486.912236       7 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


E0000 00:00:1793633489.142419       7 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


model_id = 8165f7ca | error_run_id = 4f0275a4 (IDV 29, amp 0.3)


### Adaptionsstrom

Der Strom ist auf `STEP_S` ausgedünnt. Das Driftsignal stammt aus den Daten und wird nicht rekonstruiert: `driftf` ist die Stripper-Temperatur mit Fehler bezogen auf die nominale aus *tep_generation*, punktweise exakt. Die Rücksprungstellen des Sägezahns liefert die Segmentnummer `drift_segment`; ein Kriterium auf der Differenz des verrauschten $c(t)$ träfe tausende Scheingrenzen. Die Darstellungszeit ist ein Zeitstempel ab `PLOT_EPOCH`, wie in *tep_detection* und *syn_adaptation*.

In [6]:
STRIDE = max(1, STEP_S // SAMPLE_S)
stream = error_scored.iloc[::STRIDE].copy()
N = len(stream)

X_cd = stream[feature_cols].to_numpy(dtype="float32")
y_cd = stream["cost"].to_numpy(dtype="float32")

cd_signal = stream["driftf"].to_numpy(dtype=float) - 1.0
drift_segment = stream["drift_segment"].to_numpy(dtype=float)
reset_pos = np.flatnonzero(stream["reset"].to_numpy())

t_h = np.arange(N) * STRIDE / SAMPLES_PER_H
time_index = pd.to_datetime(t_h * 3600.0, unit="s", origin=pd.Timestamp(PLOT_EPOCH))

print(f"Adaptionsstrom: {N} Punkte (STEP={STEP_S}s, STRIDE={STRIDE}, "
      f"{N * STRIDE / SAMPLES_PER_H:.0f} h), {len(reset_pos) + 1} Laeufe")
print(f"Driftsignal (gemessen): min={np.nanmin(cd_signal):.3f}, max={np.nanmax(cd_signal):.3f}")

Adaptionsstrom: 35137 Punkte (STEP=1800s, STRIDE=10, 17568 h), 1 Laeufe
Driftsignal (gemessen): min=-0.138, max=0.147


## Adaptionsstrategien

`assert_current` bricht ab, wenn `drift_adaptation.py` auf der Platte vom geladenen Modul abweicht. `SPACE_ID` und `SEL_SPACE_ID` werden aus den geladenen Code-Objekten der Suchräume gebildet und gehen zusammen mit `TPE_SETTINGS` in die Cache-Schlüssel ein. Eine geänderte Grenze oder ein anderer Sampler verwirft damit das Tuning, statt es stillschweigend zu treffen. Optuna meldet nur Fehler, den Fortschritt zeigt `status`.

In [7]:
import optuna
import tensorflow as tf
import drift_adaptation as da
from drift_adaptation import run_adaptation, StreamingDetector

optuna.logging.set_verbosity(optuna.logging.ERROR)
da.assert_current()
SPACE_ID = da.adapt_space_id()
SEL_SPACE_ID = da.selector_space_id()
print(f"Suchraum {SPACE_ID} | Selektor {SEL_SPACE_ID} | Sampler {da.TPE_SETTINGS}")

Suchraum f1f204e2 | Selektor 2a10b2ea | Sampler {'multivariate': True}


## Detektor-Konfiguration

Die Detektorparameter werden **nicht** aus *tep_detection* übernommen, sondern im geschlossenen Regelkreis mitgetunt (`_suggest_adapt_params` in `drift_adaptation.py`). Das Fehlerniveau nach einem Nachtraining unterscheidet sich vom statischen Fehlerstrom des Detektions-Notebooks, dort kalibrierte Werte sind hier systematisch fehlangepasst.

`make_detector` liefert deshalb nur die Hülle, `apply_det_params` setzt vor jedem Trial und jedem Lauf die getunten Werte. Ein Warmstart aus `tep_detection.json` wäre zudem doppelt unbrauchbar: Er überträgt Schrittzahlen vom 180-s-Rohraster auf das `STEP_TUNE_S`-Raster, und für DDM und EDDM läge sein `err_threshold` außerhalb des hier gesuchten Bereichs. Optuna übernimmt einen solchen Wert unverändert, statt ihn zu begrenzen, und verzerrt damit die erste Beobachtung.

In [8]:
def make_detector(name):
    return StreamingDetector(name)


def detector_factory(name):
    return lambda: make_detector(name)

### Normierung

Der Fehler wird mit Mittelwert und Streuung des Basismodells auf dem driftfreien Produktionsstrom normiert, wie in *tep_detection*. Die Prädiktionen liegen dort bereits als Spalte `y_pred` vor.

In [9]:
_base_free_err = (test_scored["cost"] - test_scored["y_pred"]).to_numpy(dtype=float)
NORM_MEAN = float(np.nanmean(_base_free_err))
NORM_STD = float(np.nanstd(_base_free_err)) or 1.0
print(f"Normierungs-Baseline: mean={NORM_MEAN:.4g}, std={NORM_STD:.4g}")

Normierungs-Baseline: mean=0.0005752, std=0.01314


## Raster

Tuning- und Zielraster mit ihren Chunks, die Umrechnung der Zählparameter (`_RESCALE = STRIDE_TUNE / STRIDE`) und die Refraktärzeit-Schätzung. Randsonde, Tuning und Gegenprüfungen nutzen sie gemeinsam.

In [10]:
STRIDE_TUNE = max(1, STEP_TUNE_S // SAMPLE_S)

_tune = error_scored.iloc[::STRIDE_TUNE]
X_tune = _tune[feature_cols].to_numpy(dtype="float32")
y_tune = _tune["cost"].to_numpy(dtype="float32")
_, T_STAT_TUNE = da.estimate_cooldown(_tune[["mode"]], x=COOLDOWN_X)

CHUNK_TUNE = max(1, 86400 // STEP_TUNE_S)
TUNE_FIXED = dict(chunk=CHUNK_TUNE, reject=REJECT,
                  norm_mean=NORM_MEAN, norm_std=NORM_STD, seed=SEED)
print(f"Tuningstrom: {len(X_tune)} Punkte (STRIDE_TUNE={STRIDE_TUNE}) | "
      f"chunk={CHUNK_TUNE} | T_stat(tune) ~ {T_STAT_TUNE:.0f} Punkte")

_RESCALE = STRIDE_TUNE / STRIDE
CHUNK = max(24, 86400 // STEP_S)
_BASE_CFG = dict(chunk=CHUNK, reject=REJECT, norm_mean=NORM_MEAN, norm_std=NORM_STD)
COOLDOWN, T_stat = da.estimate_cooldown(stream[["mode"]], x=COOLDOWN_X)

Tuningstrom: 5857 Punkte (STRIDE_TUNE=60) | chunk=8 | T_stat(tune) ~ 48 Punkte


## Randsonde (Phase `probe`)

Vor einem neuen Lauf wird geprüft, ob die Grenzen des vorigen Suchraums überhaupt binden. Ausgangspunkt ist der eingecheckte Endstand in `results/<NB_NAME>.json`. Für jeden Parameter einer angewandten Konfiguration, der an einer nicht natürlichen Grenze des alten Suchraums liegt (Lage ≤ 0,03 oder ≥ 0,97), treten der Grenzwert und die Werte um `PROBE_FACTORS` jenseits davon gegeneinander an, begrenzt auf den neuen Suchraum (`da.search_bounds`). Die übrigen Parameter bleiben unverändert, gerechnet wird auf dem Zielraster über `SCREENING_SEEDS`.

Eine Grenze bindet, wenn der beste Kandidat jenseits der Grenze liegt und der Grenzwert nach der Gleichstandsregel nicht gleichwertig mit ihm ist. Die Sonde ist eindimensional: Dass eine Grenze nicht bindet, ist ein Indiz, kein Beweis.

Die Zelle läuft nur mit `RUN_PHASE = "probe"`, schreibt `_temp/gates/<NB_NAME>_probe.json` und beendet das Notebook (`PhaseDone`).

In [11]:
import json
from src.utils import run_phase as rp


def _old_space(c, n_tune, epochs_hi, with_det):
    w = (5, 300, True, (False, False))
    space = {"lr": (1e-4, 1.5e-2, True, (False, False)),
             "epochs": (1, epochs_hi, False, (True, False)),
             "blind_window": (5, 400, True, (False, False)),
             "blind_period": (c, 30 * c, True, (True, False)),
             "window_prev": w, "window_post": w,
             "cooldown_x": (0.0, 60.0, False, (True, False)),
             "err_tol": (0.05, 3.0, True, (False, False)),
             "rec_horizon": (c, 365 * c, True, (True, 365 * c >= n_tune)),
             "rec_max": (15, 400, False, (False, False))}
    if with_det:
        space.update({"det_min_num_instances": (50, 400, False, (False, False)),
                      "det_alpha": (1e-6, 5e-3, True, (False, False))})
    return space


def _probe_values(p, k, side, new_space):
    low, high = new_space[k][0], new_space[k][1]
    if k == "det_min_num_instances":
        low = max(low, 2 * int(p["det_num_test_instances"]))
    out = []
    for f in (1.0, *PROBE_FACTORS):
        x = p[k] * f if side == "oben" else p[k] / f
        x = max(low, x) if high is None else min(max(low, x), high)
        x = int(round(x)) if isinstance(p[k], int) else float(x)
        if x not in out:
            out.append(x)
    return out


if RUN_PHASE == "probe":
    with open(results_dir / f"{NB_NAME}.json", encoding="utf-8") as _f:
        _old = json.load(_f)
    _old_det = _old["informed_detector"]
    _space = _old_space(CHUNK_TUNE, len(X_tune),
                        _old["tuning"].get("int_upper", {}).get("epochs", 30), _old_det == "KSWIN")
    _new_space = da.search_bounds(CHUNK_TUNE, len(X_tune), _old_det)
    _probes = []
    for _s, _p in _old["tuning"]["best_params"].items():
        if "@" in _s:
            continue
        _p = {**_p, "freeze": tuple(_p["freeze"])}
        for _e in da.edge_report(_p, _space):
            if not _e["natural"]:
                _probes.append({"strategy": _s, **_e, "base": _p,
                                "values": _probe_values(_p, _e["param"], _e["side"], _new_space)})

    _probe_cfg = base_cfg.compose(adapt_probe={
        "result": _old["_provenance"]["id"], "detector": _old_det, "step_s": STEP_S,
        "rescale": float(_RESCALE), "reject": REJECT, "chunk": CHUNK,
        "seeds": [int(s) for s in SCREENING_SEEDS],
        "tie_alpha": TIE_ALPHA, "tie_max_rel": TIE_MAX_REL,
        "probes": [{k: _q[k] for k in ("strategy", "param", "side", "values")} for _q in _probes]})
    if model_store.exists("adapt_probe", _probe_cfg) and not FORCE_RECOMPUTE:
        _probes = model_store.load("adapt_probe", _probe_cfg)
        print("Randsonde aus Cache")
    else:
        status.stage("Randsonde", total=len(_probes))
        for _q in _probes:
            _cands = {f"{_q['param']}={_v:g}": {**_q["base"], _q["param"]: _v} for _v in _q["values"]}
            _at_bound = next(iter(_cands))
            _active = da.split_strategy(_q["strategy"])[0] != "blind"
            print(f"--- {_q['strategy']}: {_q['param']} ({_q['side']}, Lage {_q['pos']:.2f}) ---")
            _, _tab = da.crosscheck_candidates(
                base_model, X_cd, y_cd, best_params={}, fixed=_BASE_CFG, t_stat=T_stat,
                detector_factory=detector_factory(_old_det) if _active else None,
                rescale=_RESCALE, mode=_q["strategy"], seeds=SCREENING_SEEDS,
                tie_alpha=TIE_ALPHA, tie_max_rel=TIE_MAX_REL, extra=_cands)
            _lead = min(_tab, key=lambda k: _tab[k]["rmse"])
            _b = _tab[_at_bound]
            _q["lead"] = _lead
            _q["binding"] = bool(_lead != _at_bound and not (
                _b["p_vs_lead"] > TIE_ALPHA and _b["rel_vs_lead"] <= TIE_MAX_REL))
            _q["table"] = {k: {kk: v[kk] for kk in ("rmse", "rmse_std", "rmse_seeds", "n_train")}
                           for k, v in _tab.items()}
            status.step(f"{_q['strategy']} {_q['param']}: "
                        f"{'bindet' if _q['binding'] else 'bindet nicht'}")
        _probes = [{k: v for k, v in _q.items() if k != "base"} for _q in _probes]
        model_store.save(_probes, "adapt_probe", _probe_cfg)
        status.done()

    _binding = any(_q["binding"] for _q in _probes)
    for _q in _probes:
        print(f"{_q['strategy']:13s} {_q['param']:22s} {_q['side']:5s} Lage {_q['pos']:.2f} -> "
              f"{'bindet' if _q['binding'] else 'bindet nicht'} (bester Kandidat {_q['lead']})")
    print(f"\nGesamt: {'mindestens eine Grenze bindet' if _binding else 'keine Grenze bindet'}")
    rp.finish_phase(base_dir, NB_NAME, "probe", {
        "binding": _binding, "result": _old["_provenance"]["id"], "probes": _probes})

## Hyperparameter-Tuning der Adaption

Getunt wird einstufig auf einem auf `STEP_TUNE_S` ausgedünnten Strom, für alle Strategien mit demselben Suchraum-Code, Budget und Sampler. Der Sampler ist ein multivariater TPE (`da.TPE_SETTINGS`). Periode und Fenster der passiven Strategie, die Fenster der Nachführphase sowie Lernrate und Epochen wirken nur gemeinsam; ein univariater TPE verfehlt schmale Optima entlang solcher Kanten. Perioden und Fenster sind logarithmisch skaliert, sodass kurze Werte wie ein tägliches Nachtraining oder ein Fenster von einem Chunk dieselbe Suchdichte erhalten wie lange.

Zielfunktion ist $J_a = \mathrm{RMSE}/\mathrm{RMSE}_\mathrm{base} + 0{,}1 \cdot n_\mathrm{train}/n_\mathrm{chunk}$ (`TUNE_WEIGHTS`). Ein Chunk entspricht einem Tag, damit Tuning- und Zielauflösung auf dieselbe Zahl an Adaptionsschritten kommen. $T_\mathrm{stat}$ wird aus den Betriebspunktwechseln geschätzt.

`_tune_case` cached jeden Fall unter seiner vollständigen Konfiguration. Der Cache muss exakt getroffen werden (ohne `rekey`): ältere Artefakte tragen weniger Schlüssel und würden sonst als Teil-Dokument übernommen. Mit `top=True` hält ein Fall zusätzlich die `TOP_K` besten, voneinander verschiedenen Trials in `top_params` fest und legt alle Trials unter `trials` ab, damit sich die Lage der Optima später ohne neue Rechnung prüfen lässt. Die Screening-Fälle brauchen sie nicht und behalten ihren bisherigen Schlüssel.

Die passive Strategie wird direkt mit vollem Budget getunt. Im Screening wird die aktive Strategie je Detektor mit reduziertem Budget getunt, ohne Warmstart, damit alle Detektoren unter denselben Bedingungen starten.

In [12]:
best_adapt_params, top_params, tune_cfgs = {}, {}, {}


def _top_trials(study, k):
    seen, out = set(), []
    for t in sorted((t for t in study.trials if t.value is not None), key=lambda t: t.value):
        p = da.decode_adapt_params(t.params)
        if rr._dumps(p) not in seen:
            seen.add(rr._dumps(p))
            out.append(p)
        if len(out) == k:
            break
    return out


def _tune_case(key, strategy, n_trials, det_name=None, enqueue=None, extra_cfg=None,
               base_params=None, top=False):
    tune_params = {"method": strategy, "case": key, "n_trials": n_trials,
                   "weights": list(TUNE_WEIGHTS), "step_tune_s": STEP_TUNE_S,
                   "step_s": STEP_S, "space": SPACE_ID, "sampler": da.TPE_SETTINGS,
                   **({"int_upper": INT_UPPER} if INT_UPPER else {}),
                   "scheme": "phase-extend", "cooldown_tuned": True,
                   **({"detector": det_name} if det_name else {}),
                   **(extra_cfg or {}),
                   **({"top_k": TOP_K} if top else {}),
                   **{k: TUNE_FIXED[k] for k in sorted(TUNE_FIXED)
                      if not k.startswith("norm_")}}
    tune_cfgs[key] = base_cfg.compose(tuning=tune_params)
    if model_store.exists("adapt_tuning", tune_cfgs[key]) and not FORCE_RECOMPUTE:
        _cached = model_store.load("adapt_tuning", tune_cfgs[key])
        best_adapt_params[key] = _cached["best"] if top else _cached
        if top:
            top_params[key] = _cached["top"]
        print(f"{key:18s} aus Cache")
        return
    status.stage(f"Tuning {key}", total=n_trials)
    tf.keras.utils.set_random_seed(SEED)
    _fac = detector_factory(det_name) if det_name else None
    _study, _ = da.tune_adaptation(strategy, base_model, X_tune, y_tune,
                                   fixed=TUNE_FIXED, detector_factory=_fac,
                                   n_trials=n_trials, weights=TUNE_WEIGHTS,
                                   seed=SEED, enqueue=enqueue, t_stat=T_STAT_TUNE,
                                   base_params=base_params,
                                   callbacks=[status.optuna_callback()],
                                   int_upper=INT_UPPER or None)
    best_adapt_params[key] = da.decode_adapt_params(_study.best_params)
    _a = _study.best_trial.user_attrs
    _msg = (f"score={_study.best_value:.3f} | rmse={_a['rmse']:.3f} "
            f"n_train={_a['n_train']:3d} n_detect={_a['n_detect']:3d}")
    print(f"{key:18s} {_msg}")
    status.done(_msg)
    if top:
        top_params[key] = _top_trials(_study, TOP_K)
        model_store.save({"best": best_adapt_params[key], "top": top_params[key],
                          "trials": da.trial_records(_study)},
                         "adapt_tuning", tune_cfgs[key])
    else:
        model_store.save(best_adapt_params[key], "adapt_tuning", tune_cfgs[key])


_tune_case("blind", "blind", N_TRIALS["final"], top=True)

print(f"\nDetektor-Screening ({N_TRIALS['screening']} Trials je Detektor)")
for _det in DETECTORS:
    _tune_case(f"informed@{_det}", "informed", N_TRIALS["screening"], det_name=_det)

[tep_adaptation] Tuning blind 0/60 | 0m00


[tep_adaptation] Tuning blind 1/60 | 5m46 | Rest ~5h40 | bester Score 0.684


[tep_adaptation] Tuning blind 3/60 | 13m03 | Rest ~4h07 | bester Score 0.684


[tep_adaptation] Tuning blind 4/60 | 15m33 | Rest ~3h37 | bester Score 0.684


[tep_adaptation] Tuning blind 5/60 | 17m17 | Rest ~3h10 | bester Score 0.672


[tep_adaptation] Tuning blind 6/60 | 17m58 | Rest ~2h41 | bester Score 0.672


... (232 outputs left out)

[tep_adaptation] Tuning informed@RMSE 38/40 | 3h22 | Rest ~10m40 | bester Score 0.451


[tep_adaptation] Tuning informed@RMSE 39/40 | 3h25 | Rest ~5m16 | bester Score 0.451


[tep_adaptation] Tuning informed@RMSE 40/40 | 3h26 | Rest ~0m00 | bester Score 0.451


informed@RMSE      score=0.451 | rmse=0.021 n_train=620 n_detect= 13
[tep_adaptation] Tuning informed@RMSE fertig nach 3h26: score=0.451 | rmse=0.021 n_train=620 n_detect= 13


## Gegenprüfung des Screenings

Die im Screening getunten Konfigurationen werden auf dem voll aufgelösten Strom über `SCREENING_SEEDS` nachgerechnet. Die Übertragung vom Tuningraster ist eine Extrapolation, und die Abstände zwischen den Detektoren liegen unter der Seed-Streuung derselben Strategie. Zählparameter werden mit `_RESCALE = STRIDE_TUNE / STRIDE` umgerechnet.

**Gleichstandsregel.** Gleichwertig mit dem Detektor mit kleinstem mittlerem RMSE ist, wer im gepaarten t-Test nicht von ihm zu trennen ist (p > `TIE_ALPHA`) und höchstens `TIE_MAX_REL` über ihm liegt. Die zweite Bedingung verhindert, dass ein Detektor mit großer Streuung allein aus fehlender Trennschärfe als gleichwertig gilt. Unter den Gleichwertigen entscheidet `TIE_ORDER`: das Verfahren mit den wenigsten Annahmen zuerst, also Schwelle auf dem Indikator, dann fehlerratenbasiert, dann fensterbasierte Tests.

Eigener Cache-Fall `adapt_screening`, exakt getroffen wie das Tuning.

In [13]:
from scipy import stats

_scr_cfg = base_cfg.compose(screening={
    "detectors": sorted(DETECTORS), "step_s": STEP_S, "cooldown_x": COOLDOWN_X,
    "space": SPACE_ID, "rescale": _RESCALE, "reject": REJECT,
    "seeds": [int(s) for s in SCREENING_SEEDS],
    "params": {d: da.encode_adapt_params(best_adapt_params[f"informed@{d}"])
               for d in DETECTORS}})

if model_store.exists("adapt_screening", _scr_cfg) and not FORCE_RECOMPUTE:
    screening = model_store.load("adapt_screening", _scr_cfg)
    print("Screening aus Cache")
else:
    screening = {}
    status.stage("Screening-Gegenpruefung", total=len(DETECTORS) * len(SCREENING_SEEDS))
    for _det in DETECTORS:
        _key = f"informed@{_det}"
        _rmse, _ntr = [], []
        for _sd in SCREENING_SEEDS:
            _, _tab = da.crosscheck_candidates(
                base_model, X_cd, y_cd,
                best_params={"informed": best_adapt_params[_key]}, fixed=_BASE_CFG,
                t_stat=T_stat, detector_factory=detector_factory(_det),
                rescale=_RESCALE, mode="informed", seed=int(_sd), verbose=False)
            _rmse.append(float(_tab["tpe_best"]["rmse"]))
            _ntr.append(int(_tab["tpe_best"]["n_train"]))
            status.step(f"{_det} Seed {_sd}")
        screening[_det] = {"rmse": float(np.mean(_rmse)), "rmse_std": float(np.std(_rmse, ddof=1)),
                           "rmse_seeds": _rmse, "n_train": float(np.mean(_ntr))}
        print(f"{_det:6s} RMSE={screening[_det]['rmse']:.4f} "
              f"+/- {screening[_det]['rmse_std']:.4f}  "
              f"({', '.join(f'{r:.3f}' for r in _rmse)})  "
              f"n_train={screening[_det]['n_train']:.0f}")
    model_store.save(screening, "adapt_screening", _scr_cfg)

_lead = min(screening, key=lambda d: screening[d]["rmse"])
_tied = []
for _det in DETECTORS:
    if _det == _lead:
        _tied.append(_det)
        continue
    _p = stats.ttest_rel(screening[_det]["rmse_seeds"], screening[_lead]["rmse_seeds"]).pvalue
    _rel = screening[_det]["rmse"] / (screening[_lead]["rmse"] or np.nan) - 1.0
    screening[_det]["p_vs_lead"] = float(_p)
    screening[_det]["rel_vs_lead"] = float(_rel)
    if _p > TIE_ALPHA and _rel <= TIE_MAX_REL:
        _tied.append(_det)
screening[_lead]["p_vs_lead"] = 1.0
screening[_lead]["rel_vs_lead"] = 0.0
BEST_DETECTOR = min(_tied, key=TIE_ORDER.index)
SCREEN_LEAD = _lead
SCREEN_TIED = tuple(_tied)

_msg = (f"kleinster RMSE {_lead}, gleichwertig {_tied}, gewaehlt {BEST_DETECTOR} "
        f"(RMSE={screening[BEST_DETECTOR]['rmse']:.4f} "
        f"+/- {screening[BEST_DETECTOR]['rmse_std']:.4f})")
print(f"\n{_msg}")
status.done(_msg, notify=status.label is not None)

screening_table = pd.DataFrame(
    {d: {"rmse": v["rmse"], "rmse_std": v["rmse_std"], "p_vs_lead": v["p_vs_lead"],
         "rel_vs_lead": v["rel_vs_lead"], "n_train": v["n_train"]}
     for d, v in screening.items()}).T

[tep_adaptation] Screening-Gegenpruefung 0/25 | 0m00


[tep_adaptation] Screening-Gegenpruefung 1/25 | 7m17 | Rest ~2h54 | KSWIN Seed 42


[tep_adaptation] Screening-Gegenpruefung 2/25 | 14m40 | Rest ~2h48 | KSWIN Seed 92


[tep_adaptation] Screening-Gegenpruefung 3/25 | 21m31 | Rest ~2h37 | KSWIN Seed 791


[tep_adaptation] Screening-Gegenpruefung 4/25 | 28m27 | Rest ~2h29 | KSWIN Seed 669


[tep_adaptation] Screening-Gegenpruefung 5/25 | 35m00 | Rest ~2h20 | KSWIN Seed 449


... (22 outputs left out)

[tep_adaptation] Screening-Gegenpruefung 23/25 | 5h08 | Rest ~26m50 | RMSE Seed 791


[tep_adaptation] Screening-Gegenpruefung 24/25 | 5h26 | Rest ~13m37 | RMSE Seed 669


[tep_adaptation] Screening-Gegenpruefung 25/25 | 5h44 | Rest ~0m00 | RMSE Seed 449


RMSE   RMSE=0.0210 +/- 0.0010  (0.022, 0.021, 0.020, 0.020, 0.021)  n_train=622

kleinster RMSE ADWIN, gleichwertig ['ADWIN', 'RMSE'], gewaehlt RMSE (RMSE=0.0210 +/- 0.0010)
[tep_adaptation] Screening-Gegenpruefung fertig nach 5h44: kleinster RMSE ADWIN, gleichwertig ['ADWIN', 'RMSE'], gewaehlt RMSE (RMSE=0.0210 +/- 0.0010)


## Finales Tuning und Gegenprüfung

Mit dem gewählten Detektor wird die aktive Strategie mit vollem Budget nachgetunt, Warmstart ist das Screening-Optimum. Danach folgt die kombinierte Strategie, Warmstart ist die Vereinigung der Einzeloptima von passiv und aktiv.

Die Gegenprüfung behandelt alle Strategien gleich. Je Strategie treten die `TOP_K` besten, voneinander verschiedenen Trials ihrer eigenen Studie auf dem voll aufgelösten Strom über `SCREENING_SEEDS` an. Entschieden wird nach der Gleichstandsregel des Screenings, unter Gleichwertigen gewinnt der Kandidat mit weniger Nachtrainings. Das ist derselbe Nebenaspekt, den die Zielfunktion mit 0,1 gewichtet. Kandidaten aus anderen Strategien treten nicht an, damit keine Strategie mehr Auswahl hat als eine andere.

Eigener Cache-Fall `adapt_crosscheck`, dessen Schlüssel an allen Kandidaten hängt. Er schreibt nicht nach `adapt_tuning` zurück, sonst startete der nächste Lauf von anderen Kandidaten. `CC_WON` hält fest, welcher Rang je Strategie gewonnen hat.

In [14]:
_scr = best_adapt_params[f"informed@{BEST_DETECTOR}"]
_tune_case("informed", "informed", N_TRIALS["final"], det_name=BEST_DETECTOR,
           enqueue=[da.encode_adapt_params(_scr)], top=True)

_bl, _inf = best_adapt_params["blind"], best_adapt_params["informed"]
_tune_case("combined", "combined", N_TRIALS["final"], det_name=BEST_DETECTOR,
           enqueue=[da.encode_adapt_params({**_bl, **_inf}),
                    da.encode_adapt_params({**_inf, **_bl})], top=True)

CC_WON = {}


def _crosscheck(mode, candidates, det_name):
    _best, _tab = da.crosscheck_candidates(
        base_model, X_cd, y_cd, best_params={}, fixed=_BASE_CFG, t_stat=T_stat,
        detector_factory=detector_factory(det_name) if det_name else None,
        rescale=_RESCALE, mode=mode, seed=SEED, seeds=SCREENING_SEEDS,
        tie_alpha=TIE_ALPHA, tie_max_rel=TIE_MAX_REL,
        extra={f"top{i + 1}": p for i, p in enumerate(candidates)})
    return _best, next(k for k, v in _tab.items() if v["params"] == _best)


def _crosscheck_case(case, keys, candidates, det_names):
    _cfg = base_cfg.compose(**{case: {
        "detector": BEST_DETECTOR, "space": SPACE_ID, "step_s": STEP_S,
        "cooldown_x": COOLDOWN_X, "rescale": float(_RESCALE), "reject": REJECT, "chunk": CHUNK,
        "seeds": [int(s) for s in SCREENING_SEEDS],
        "tie_alpha": TIE_ALPHA, "tie_max_rel": TIE_MAX_REL, "top_k": TOP_K,
        "candidates": {k: [da.encode_adapt_params(p) for p in candidates[k]] for k in keys}}})
    if model_store.exists(f"adapt_{case}", _cfg) and not FORCE_RECOMPUTE:
        _res = model_store.load(f"adapt_{case}", _cfg)
        print(f"Gegenpruefung {case} aus Cache")
    else:
        _res = {}
        status.stage(f"Gegenpruefung {', '.join(keys)}", total=len(keys))
        for k in keys:
            print(f"--- {k} ---")
            _p, _won = _crosscheck(k, candidates[k], det_names[k])
            _res[k] = {"params": _p, "won": _won}
            status.step(f"{k}: {_won}")
        model_store.save(_res, f"adapt_{case}", _cfg)
        status.done(", ".join(f"{k}: {v['won']}" for k, v in _res.items()), notify=True)
    for k, v in _res.items():
        best_adapt_params[k], CC_WON[k] = v["params"], v["won"]
        print(f"  {k:13s} {v['won']}: {v['params']}")


_crosscheck_case("crosscheck", ("blind", "informed", "combined"), top_params,
                 {"blind": None, "informed": BEST_DETECTOR, "combined": BEST_DETECTOR})

[tep_adaptation] Tuning informed 0/60 | 0m00


[tep_adaptation] Tuning informed 1/60 | 14m19 | Rest ~14h05 | bester Score 0.451


[tep_adaptation] Tuning informed 2/60 | 19m30 | Rest ~9h25 | bester Score 0.451


[tep_adaptation] Tuning informed 3/60 | 20m07 | Rest ~6h22 | bester Score 0.451


[tep_adaptation] Tuning informed 6/60 | 22m18 | Rest ~3h20 | bester Score 0.451


[tep_adaptation] Tuning informed 7/60 | 23m19 | Rest ~2h56 | bester Score 0.451


... (125 outputs left out)

top2           RMSE=0.0204 +/- 0.0002  n_train=627  n_detect=55


top3           RMSE=0.0197 +/- 0.0008  n_train=687  n_detect=57
kleinster RMSE: top1; gleichwertig (p > 0.05 und hoechstens 5 % darueber): ['top1']
uebernommen: top1 (RMSE=0.019 +/- 0.001, n_train=696)
[tep_adaptation] Gegenpruefung blind, informed, combined 3/3 | 17h02 | Rest ~0m00 | combined: top1


[tep_adaptation] Gegenpruefung blind, informed, combined fertig nach 17h02: blind: top1, informed: top2, combined: top1


  blind         top1: {'lr': 0.0004566767007755255, 'epochs': 36, 'freeze': (False, False, False), 'reset': False, 'blind_window': 655, 'blind_period': 9}
  informed      top2: {'lr': 0.00036192275196490883, 'epochs': 52, 'freeze': (False, False, False), 'reset': False, 'window_prev': 33, 'window_post': 705, 'cooldown_x': 12.099949707396366, 'det_window': 51, 'det_threshold': 2.501149358626852}
  combined      top1: {'lr': 0.00023265970630704304, 'epochs': 60, 'freeze': (False, False, False), 'reset': False, 'blind_window': 59, 'blind_period': 14, 'window_prev': 25, 'window_post': 595, 'cooldown_x': 8.30740601505208, 'det_window': 55, 'det_threshold': 1.0858508993132971}


## Selektorparameter

Der Recurrence-Selektor ergänzt das Nachtrainingsfenster um vergangene Punkte mit ähnlichem Fehler. Für alle drei Strategien bleiben die gegengeprüften Strategieparameter fest, getunt werden nur `err_tol`, `rec_horizon` und `rec_max` (`base_params` in `tune_adaptation`, Suchraum `SEL_SPACE_ID`). Der Unterschied zur Strategie ohne Selektor geht damit allein auf die Datenauswahl zurück. Die Gegenprüfung folgt demselben Schema wie oben, mit den `TOP_K` besten Selektorsätzen auf der festen Basis (Cache-Fall `adapt_crosscheck_rec`).

In [15]:
REC_STRATEGIES = ("blind", "informed", "combined")
SEL_KEYS = ("err_tol", "rec_horizon", "rec_max")

_rec_cands, _rec_det = {}, {}
for _s in REC_STRATEGIES:
    _key = f"{_s}_rec"
    _base = dict(best_adapt_params[_s])
    _tune_case(_key, _key, REC_TRIALS, det_name=None if _s == "blind" else BEST_DETECTOR,
               extra_cfg={"base": da.encode_adapt_params(_base), "selector_space": SEL_SPACE_ID},
               base_params=_base, top=True)
    _rec_cands[_key] = [{**_base, **{k: p[k] for k in SEL_KEYS}} for p in top_params[_key]]
    _rec_det[_key] = None if _s == "blind" else BEST_DETECTOR

_crosscheck_case("crosscheck_rec", tuple(_rec_cands), _rec_cands, _rec_det)

[tep_adaptation] Tuning blind_rec 0/30 | 0m00


[tep_adaptation] Tuning blind_rec 1/30 | 9m43 | Rest ~4h42 | bester Score 0.469


[tep_adaptation] Tuning blind_rec 2/30 | 19m34 | Rest ~4h33 | bester Score 0.469


[tep_adaptation] Tuning blind_rec 3/30 | 29m37 | Rest ~4h26 | bester Score 0.469


[tep_adaptation] Tuning blind_rec 4/30 | 39m26 | Rest ~4h16 | bester Score 0.469


[tep_adaptation] Tuning blind_rec 5/30 | 49m06 | Rest ~4h05 | bester Score 0.469


... (101 outputs left out)

top2           RMSE=0.0174 +/- 0.0005  n_train=693  n_detect=47


top3           RMSE=0.0181 +/- 0.0012  n_train=689  n_detect=49
kleinster RMSE: top2; gleichwertig (p > 0.05 und hoechstens 5 % darueber): ['top2', 'top1', 'top3']
uebernommen: top3 (RMSE=0.018 +/- 0.001, n_train=689)
[tep_adaptation] Gegenpruefung blind_rec, informed_rec, combined_rec 3/3 | 19h10 | Rest ~0m00 | combined_rec: top3


[tep_adaptation] Gegenpruefung blind_rec, informed_rec, combined_rec fertig nach 19h10: blind_rec: top1, informed_rec: top3, combined_rec: top3


  blind_rec     top1: {'lr': 0.0004566767007755255, 'epochs': 36, 'freeze': (False, False, False), 'reset': False, 'blind_window': 655, 'blind_period': 9, 'err_tol': 10.49898085146362, 'rec_horizon': 4029, 'rec_max': 28}
  informed_rec  top3: {'lr': 0.00036192275196490883, 'epochs': 52, 'freeze': (False, False, False), 'reset': False, 'window_prev': 33, 'window_post': 705, 'cooldown_x': 12.099949707396366, 'det_window': 51, 'det_threshold': 2.501149358626852, 'err_tol': 3.2435341628194965, 'rec_horizon': 5746, 'rec_max': 3}
  combined_rec  top3: {'lr': 0.00023265970630704304, 'epochs': 60, 'freeze': (False, False, False), 'reset': False, 'blind_window': 59, 'blind_period': 14, 'window_prev': 25, 'window_post': 595, 'cooldown_x': 8.30740601505208, 'det_window': 55, 'det_threshold': 1.0858508993132971, 'err_tol': 0.4896142982923554, 'rec_horizon': 3202, 'rec_max': 1}


## Prüfung vor den Seed-Läufen (Phase `tune`)

Nach Tuning, Gegenprüfungen und Selektor-Tuning, vor dem teuren Hauptlauf. Beide Bedingungen müssen erfüllt sein:

1. **Keine willkürliche Grenze.** Kein Parameter einer angewandten Konfiguration liegt an einer Grenze von `da.search_bounds`, die nicht als natürlich markiert ist (Lage ≤ 0,03 oder ≥ 0,97).
2. **Keine Verdünnung.** Für die Studien mit vollem Budget (passiv, aktiv, kombiniert) ist der beste Trial auf dem Tuningraster höchstens `TIE_MAX_REL` schlechter als die angewandte Konfiguration des vorigen Laufs, die im neuen Suchraum enthalten ist. Beide werden mit derselben Zielfunktion und demselben Seed nachgerechnet. Bei anderem Detektor entfällt der Vergleich für die aktiven Strategien.

Die Zelle läuft nur mit `RUN_PHASE = "tune"`, schreibt `_temp/gates/<NB_NAME>_tune.json` und beendet das Notebook. In Phase `full` kommen Tuning und Gegenprüfungen aus dem Cache.

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [16]:
import json
from src.utils import run_phase as rp

GATE_STRATEGIES = ("blind", "informed", "combined", "blind_rec", "informed_rec", "combined_rec")

if RUN_PHASE == "tune":
    _edges = {}
    for _s in GATE_STRATEGIES:
        _det = None if da.split_strategy(_s)[0] == "blind" else BEST_DETECTOR
        _bounds = da.search_bounds(CHUNK_TUNE, len(X_tune), _det)
        _edges[_s] = [_e for _e in da.edge_report(best_adapt_params[_s], _bounds)
                      if not _e["natural"]]

    with open(results_dir / f"{NB_NAME}.json", encoding="utf-8") as _f:
        _old = json.load(_f)
    _base_res = run_adaptation(base_model, X_tune, y_tune, mode="baseline", **TUNE_FIXED)
    _rmse_base_tune = float(np.sqrt(np.mean(np.asarray(_base_res["errors"], float) ** 2)))
    _dilution = {}
    status.stage("Pruefung Verduennung", total=3)
    for _s in ("blind", "informed", "combined"):
        _active = _s != "blind"
        if _active and _old["informed_detector"] != BEST_DETECTOR:
            _dilution[_s] = {"skipped": f"Detektor {_old['informed_detector']} -> {BEST_DETECTOR}"}
            status.step(_s)
            continue
        _po = {**_old["tuning"]["best_params"][_s]}
        _po["freeze"] = tuple(_po["freeze"])
        _sc = {k: da.score_params(_s, base_model, X_tune, y_tune, p, fixed=TUNE_FIXED,
                                  t_stat=T_STAT_TUNE, weights=TUNE_WEIGHTS,
                                  detector_factory=detector_factory(BEST_DETECTOR) if _active else None,
                                  rmse_baseline=_rmse_base_tune, seed=SEED)["score"]
               for k, p in (("old", _po), ("new", top_params[_s][0]))}
        _dilution[_s] = {**_sc, "diluted": bool(_sc["new"] > _sc["old"] * (1 + TIE_MAX_REL))}
        status.step(_s)
    status.done()

    _pass = (not any(_edges.values())
             and not any(_d.get("diluted", False) for _d in _dilution.values()))
    for _s, _rows in _edges.items():
        for _e in _rows:
            print(f"Rand: {_s:13s} {_e['param']:22s} {_e['side']:5s} Lage {_e['pos']:.2f}")
    for _s, _d in _dilution.items():
        print(f"Verduennung {_s:9s} {_d}")
    print("Pruefung bestanden" if _pass else "Pruefung NICHT bestanden")
    rp.finish_phase(base_dir, NB_NAME, "tune", {
        "pass": _pass, "detector": BEST_DETECTOR, "edges": _edges, "dilution": _dilution})

[tep_adaptation] Pruefung Verduennung 0/3 | 0m00


[tep_adaptation] Pruefung Verduennung 1/3 | 19m05 | Rest ~38m09 | blind


[tep_adaptation] Pruefung Verduennung 3/3 | 19m05 | Rest ~0m00 | combined


[tep_adaptation] Pruefung Verduennung fertig nach 19m05


Rand: combined      epochs                 oben  Lage 1.00
Rand: combined_rec  epochs                 oben  Lage 1.00
Verduennung blind     {'old': 0.4792153528296581, 'new': 0.4639287132243147, 'diluted': False}
Verduennung informed  {'skipped': 'Detektor KSWIN -> RMSE'}
Verduennung combined  {'skipped': 'Detektor KSWIN -> RMSE'}
Pruefung NICHT bestanden


PhaseDone: tep_adaptation: Phase tune beendet, Gate /work/_temp/gates/tep_adaptation_tune.json

## Adaption ausführen

Ein Adaptionslauf hängt nur von Modus, angewandter Konfiguration und Seed ab. `reuse_runs` übernimmt deshalb Einzelläufe aus Archiven desselben Falls, deren Dokument sich nur in den angewandten Konfigurationen anderer Strategien unterscheidet. Eine geänderte Strategie löst so nicht das Neurechnen aller übrigen aus.

In [ ]:
import json
from glob import glob


def _split_doc(doc, section):
    d = json.loads(rr._dumps(doc))
    applied = d.get("adaptation", {}).pop("applied", {})
    items = applied if section == "adaptation" else d.pop(section, {})
    return rr._dumps(d), items


def reuse_runs(case, cfg, section, names):
    rest, items = _split_doc(cfg.document, section)
    want = {}
    for n in names:
        want.setdefault(rr._dumps([n, items[n]]), []).append(n)
    found = {}
    for _path in sorted(glob(os.path.join(model_dir, f"{model_store.prefix}_{case}_*.json"))):
        if len(found) == len(names):
            break
        with open(_path, encoding="utf-8") as _f:
            _sc = json.load(_f)
        if _sc.get("case") != case or _sc.get("id") == cfg.id:
            continue
        _rest, _items = _split_doc(_sc["config"], section)
        if _rest != rest:
            continue
        hits = {n: o for o, a in _items.items()
                for n in want.get(rr._dumps([o, a]), []) if n not in found}
        if hits:
            _res = model_store.load(case, run_id=_sc["id"], rename=True)
            found.update({n: _res[o] for n, o in hits.items() if o in _res})
    return found

`ADAPT_CFG` enthält die Vorgaben, die das Tuning überschreibt. `applied_cfg` ist die angewandte Konfiguration je Strategie in Zielauflösung: Zählparameter über `scale_adapt_params` umgerechnet, `cooldown_x` über $T_\mathrm{stat}$ in `cooldown` übersetzt. Der Cache-Fall `adaptation` wird exakt getroffen; ein Eintrag, dem eine Strategie fehlt, wird neu gerechnet.

In [ ]:
ADAPT_CFG = dict(
    lr=3e-3, epochs=9, freeze=(True, False, False), reset=False,
    blind_window=44, blind_period=CHUNK,
    window_prev=max(1, 300 // STRIDE), window_post=max(1, 300 // STRIDE),
    reject=REJECT, chunk=CHUNK, norm_mean=NORM_MEAN, norm_std=NORM_STD,
)
STRATEGIES = ("baseline", "blind", "informed", "combined",
              "blind_rec", "informed_rec", "combined_rec")
ACTIVE = {"informed", "combined", "informed_rec", "combined_rec"}

applied_cfg = {s: da.resolve_cooldown(
                   {**ADAPT_CFG, **da.scale_adapt_params(best_adapt_params.get(s, {}), _RESCALE)},
                   T_stat)
               for s in STRATEGIES}
print(f"T_stat ~ {T_stat:.0f} Punkte (~{T_stat * STEP_S / 3600:.1f} h) | "
      f"Detektor = {BEST_DETECTOR}")

adapt_cfg = base_cfg.compose(adaptation={
    "step_s": STEP_S, "detector": BEST_DETECTOR, "scheme": "phase-extend",
    "cooldown_tuned": True, "screening": sorted(DETECTORS),
    "applied": {k: {kk: v for kk, v in applied_cfg[k].items()
                    if kk not in ("norm_mean", "norm_std")} for k in STRATEGIES}})


def _run(name, seed):
    tf.keras.utils.set_random_seed(seed)
    _det = make_detector(BEST_DETECTOR) if name in ACTIVE else None
    _det, _cfg = da.apply_det_params(_det, applied_cfg[name])
    return run_adaptation(base_model, X_cd, y_cd, mode=name, detector=_det, seed=seed, **_cfg)


_adapt_from_cache = model_store.exists("adaptation", adapt_cfg) and not FORCE_RECOMPUTE
if _adapt_from_cache:
    results = model_store.load("adaptation", adapt_cfg, rename=True)
    _missing = [s for s in STRATEGIES if s not in results]
    if _missing:
        print(f"Cache-Eintrag ohne Strategien {_missing} -- wird neu gerechnet.")
        _adapt_from_cache = False
if not _adapt_from_cache:
    results = reuse_runs("adaptation", adapt_cfg, "adaptation", STRATEGIES)
    if results:
        print(f"Aus Archiven uebernommen: {sorted(results)}")
    status.stage("Hauptlauf", total=len(STRATEGIES) - len(results))
    for _name in STRATEGIES:
        if _name in results:
            continue
        print(f"Strategie '{_name}' ...")
        results[_name] = _run(_name, SEED)
        print(f"  -> {len(results[_name]['train_steps'])} Nachtrainings, "
              f"{len(results[_name]['detect_steps'])} Detektionen")
        status.step(_name)
    model_store.save(results, "adaptation", adapt_cfg)
    status.done()

## Seed-Robustheit — Läufe und repräsentativer Seed

Alle Strategien laufen über `N_SEED_RUNS` Seeds. Die Zielliste steht vollständig fest, bevor gerechnet wird; sie hängt allein von `SEED` und `N_SEED_RUNS` ab, nicht davon, was im Archiv liegt. Läufe mit derselben Konfiguration und anderem Seed werden aus dem Archiv geladen (Vergleich über die Kennung ohne `seed_eval`-Sektion), fehlende nachgerechnet. Jeder Seed wird einzeln gespeichert, ein Abbruch verliert höchstens den laufenden.

Der Medoid-Seed liefert die exemplarischen Zeitreihen: Sein RMSE-Vektor über alle Strategien liegt, je Strategie standardisiert, dem Mittel über alle Seeds am nächsten. So repräsentiert die eingebundene Grafik den Gesamttrend statt eines festen Seeds.

In [ ]:
_case = "adaptation_seeds"

seed_results = {SEED: results}
for _sc_path in sorted(glob(os.path.join(model_dir, f"{model_store.prefix}_{_case}_*.json"))):
    with open(_sc_path, encoding="utf-8") as _f:
        _sc = json.load(_f)
    if _sc.get("case") != _case:
        continue
    _doc = dict(_sc["config"])
    _seed = (_doc.pop("seed_eval", None) or {}).get("seed")
    if _seed is None or _seed in seed_results:
        continue
    if rr.RunConfig.from_document(_doc).id != adapt_cfg.id:
        continue
    seed_results[_seed] = model_store.load(_case, run_id=_sc["id"], rename=True)
print(f"{len(seed_results) - 1} Seeds aus dem Archiv")

_rng_seed = np.random.default_rng(SEED)
TARGET_SEEDS = [SEED]
while len(TARGET_SEEDS) < N_SEED_RUNS:
    _s = int(_rng_seed.integers(1, 2**10 - 1))
    if _s not in TARGET_SEEDS:
        TARGET_SEEDS.append(_s)
_todo = [s for s in TARGET_SEEDS if s not in seed_results]
if _todo:
    status.stage("Seeds", total=len(_todo))
for _seed in _todo:
    _cfg_seed = adapt_cfg.compose(seed_eval={"seed": _seed})
    _res = reuse_runs(_case, _cfg_seed, "adaptation", STRATEGIES)
    for _name in STRATEGIES:
        if _name not in _res:
            _res[_name] = _run(_name, _seed)
    model_store.save(_res, _case, _cfg_seed)
    seed_results[_seed] = _res
    status.step(f"Seed {_seed}")
if _todo:
    status.done(f"{len(seed_results)} Seeds", notify=True)

SEEDS = tuple(seed_results)
seed_rmse = pd.DataFrame(
    {_sd: {s: float(np.sqrt(np.mean(np.asarray(_r[s]["errors"], float) ** 2)))
           for s in STRATEGIES}
     for _sd, _r in seed_results.items()})
seed_rmse["mean"] = seed_rmse[list(seed_results)].mean(axis=1)
seed_rmse["std"] = seed_rmse[list(seed_results)].std(axis=1)
print(f"\n{len(seed_results)} Laeufe")
print(seed_rmse[["mean", "std"]].to_string(float_format=lambda v: f"{v:.4f}"))

In [ ]:
_seed_cols = list(seed_results)
_M = seed_rmse.loc[list(STRATEGIES), _seed_cols]
_z = _M.sub(_M.mean(axis=1), axis=0).div(_M.std(axis=1).replace(0, 1), axis=0)
_medoid = (_z ** 2).sum(axis=0).pow(0.5).idxmin()
print(f"Repraesentativer (Medoid-)Seed: {_medoid}  (von {len(_seed_cols)} Seeds)")

## Vergleich der Adaptionsstrategien

Je Strategie der normierte Modellfehler auf symlog-Ordinate, das gemessene Driftsignal $c(t)$ auf der rechten Achse und die Detektionszeitpunkte. Dargestellt sind die ersten `PLOT_MONTHS` Monate des Medoid-Seeds; die Kennzahlen weiter unten beziehen sich auf den vollständigen Strom und auf alle Seeds. Die Rücksprungstellen `cut` für den RMSE-Verlauf kommen aus `drift_segment`.

Die Abbildungen werden immer geschrieben, auch aus dem Cache, sonst blieben die stabilen `.pgf` auf dem Stand eines früheren Laufs, während JSON und TeX schon die neue Kennung tragen.

In [ ]:
import matplotlib.pyplot as plt
from src.utils import thesis_style as ts

_mask_plot = (np.ones(N, bool) if PLOT_MONTHS is None else np.asarray(
    time_index < pd.Timestamp(PLOT_EPOCH) + pd.DateOffset(months=PLOT_MONTHS)))
t_plot = time_index[_mask_plot]
cd_plot = cd_signal[_mask_plot]
n_plot = len(t_plot)
cut = np.flatnonzero(np.diff(drift_segment[_mask_plot]) > 0) + 1

rng_plot = np.random.default_rng(SEED)

PANELS = [("baseline", "Baseline"), ("blind", "Blind"), ("informed", "Informed"),
          ("combined", "Combined"), ("blind_rec", "Blind+Rec")]

for k, (key, title) in enumerate(PANELS):
    res = seed_results[_medoid][key]
    role = "drift_afflicted" if key == "baseline" else "adapted"
    label = "drift-behaftet" if key == "baseline" else "adaptiert"

    fig, ax, ax2 = ts.error_timeseries(
        t_plot,
        [(np.asarray(res["errors_norm"], float)[_mask_plot], role, label)],
        cd=cd_plot,
        detections=[t_plot[d] for d in res["detect_steps"] if d < n_plot],
        xlabel="Zeit $t$ [Monate]" if k == len(PANELS) - 1 else None,
        cd_label="Driftsignal $c(t)$",
        ylim=PLOT_YLIM, cd_amp=0.5, cd_tick_step=0.2,
        frac=PLOT_FRAC, rng=rng_plot,
        legend_series=[("drift_afflicted", "drift-behaftet"), ("adapted", "adaptiert")],
        legend=dict(bbox_to_anchor=[0.49, 1.15], loc="upper center", ncols=3) if k == 0 else False,
    )
    fig.tight_layout()
    plot_store.save_figure(fig, f"adaptation_{key}", adapt_cfg, final=IS_FINAL,
                           savefig_kwargs={"dpi": 300}, archive_kwargs={"dpi": 300})
    plt.show()

## RMSE-Verlauf der Strategien

In [ ]:
from matplotlib import dates as mdates
from src.utils import thesis_style as ts

WIN = max(10, 86400 // STEP_S)

fig, ax = plt.subplots(figsize=(ts.fig_width(), ts.fig_width(0.45)))

_adapt_labeled = False
for key, title in PANELS:
    e = pd.Series(np.asarray(seed_results[_medoid][key]["errors"], float), index=time_index)
    rmse_roll = np.sqrt((e ** 2).rolling(WIN, min_periods=1).mean())
    rmse_roll = rmse_roll[_mask_plot]
    if key == "baseline":
        _col, _lab = ts.C["drift_afflicted"], "drift-behaftet"
    else:
        _col = ts.C["adapted"]
        _lab = "adaptiert" if not _adapt_labeled else None
        _adapt_labeled = True
    ax.plot(rmse_roll.index, rmse_roll.values, color=_col, linewidth=1.4, label=_lab)

ax2 = ax.twinx()
for i, (a, b) in enumerate(zip(np.r_[0, cut], np.r_[cut, len(cd_plot)])):
    ax2.plot(t_plot[a:b], cd_plot[a:b], **ts.line("drift_signal", linewidth=1.0))
ax2.set_ylabel("Driftsignal $c(t)$")
ax2.spines["right"].set_visible(True)
ax2.spines["top"].set_visible(False)

ax.set_ylabel("Gleitender RMSE")
ax.set_xlabel("Zeit $t$ [Monate]")
ax.margins(x=0)
ax.grid(True, alpha=0.2)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.legend(loc="upper left", ncol=2, frameon=False)
fig.tight_layout()
plt.show()

## Numerische Auswertung

Kennzahlen je Seed (RMSE-Verbesserung seed-intern gegen die eigene Baseline), danach Mittelwert und Streuung über alle Seeds. `stats_mean` treibt Ergebnisdatei und Makros, die Abbildungen zeigen den Medoid-Seed.

In [ ]:
def _strategy_stats(res, c, rmse_baseline):
    e = np.asarray(res["errors"], float)
    c = np.asarray(c, float)
    rmse = float(np.sqrt(np.mean(e ** 2)))
    return dict(
        rmse=rmse,
        bias=float(np.mean(e)),
        sigma=float(np.std(e)),
        corr_abs=float(np.corrcoef(np.abs(e), np.abs(c))[0, 1]),
        n_train=int(len(res["train_steps"])),
        n_detect=int(len(res["detect_steps"])),
        n_reject=int(len(res.get("reject_steps", []))),
        n_recall=float(np.mean(res.get("recall_counts") or [0.0])),
        rmse_improvement=float(1.0 - rmse / rmse_baseline) if rmse_baseline else 0.0,
    )


def _rmse(res):
    return float(np.sqrt(np.mean(np.asarray(res["errors"], float) ** 2)))


_METRIC_COLS = ["rmse", "bias", "sigma", "corr_abs",
                "n_train", "n_detect", "n_reject", "n_recall", "rmse_improvement"]

_per_seed = {sd: {k: _strategy_stats(r[k], cd_signal, _rmse(r["baseline"])) for k in STRATEGIES}
             for sd, r in seed_results.items()}
_stack = {k: pd.DataFrame([_per_seed[sd][k] for sd in seed_results])[_METRIC_COLS]
          for k in STRATEGIES}
stats_mean = pd.DataFrame({k: _stack[k].mean() for k in STRATEGIES}).T[_METRIC_COLS]
stats_std = pd.DataFrame({k: _stack[k].std(ddof=1) for k in STRATEGIES}).T[_METRIC_COLS]
stats = stats_mean

pd.set_option("display.float_format", lambda v: f"{v:.4g}")
print(f"Kennzahlen als Mittelwert ueber {len(seed_results)} Seeds:")
print(stats_mean.to_string())
print("\nStreuung (Std ueber Seeds):")
print(stats_std.to_string())

## Wirkung des Selektors

Gepaart über dieselben Seeds: RMSE mit gegen ohne Selektor je Strategie, mittlere Differenz, gepaarter t-Test und Zahl der Seeds mit Verbesserung.

In [ ]:
from scipy.stats import ttest_rel

_rows = {}
for _s in REC_STRATEGIES:
    _r = f"{_s}_rec"
    _without = np.array([_rmse(seed_results[sd][_s]) for sd in seed_results])
    _with = np.array([_rmse(seed_results[sd][_r]) for sd in seed_results])
    _d = _with - _without
    _rows[_r] = {"rmse": _with.mean(), "rmse_std": _with.std(ddof=1),
                 "rmse_without": _without.mean(), "rmse_without_std": _without.std(ddof=1),
                 "delta": _d.mean(), "delta_std": _d.std(ddof=1),
                 "delta_rel": _d.mean() / _without.mean(),
                 "p_value": ttest_rel(_with, _without).pvalue,
                 "n_better": int((_d < 0).sum()),
                 "n_train": stats_mean.loc[_r, "n_train"],
                 "n_recall": stats_mean.loc[_r, "n_recall"]}
rec_table = pd.DataFrame(_rows).T

print(f"Selektor gepaart ueber {len(seed_results)} Seeds (Detektor {BEST_DETECTOR}):")
print(rec_table.to_string(float_format=lambda v: f"{v:.4g}"))

## Seed-Robustheit — Verteilungsplot

Je Strategie eine längs geteilte Violine über alle Seeds, links ohne Selektor (aufgehellt), rechts mit. Die Boxplots sind ebenso längs geteilt, Median und Whisker liegen auf der Trennlinie; gleiche Verteilungen ergeben so wieder einen gewöhnlichen Boxplot. Farben je Strategie aus `ts.SET2`, die Legende ist neutral grau.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils import thesis_style as ts

SPLIT_LABELS = {"blind": "Passiv", "informed": "Aktiv", "combined": "Kombiniert"}
SEL_LABELS = ("ohne Selektor", "mit Selektor")
STRAT_COLORS = dict(zip(REC_STRATEGIES, ts.SET2))
LIGHTEN = 0.45


def _light(c):
    return tuple(LIGHTEN * ch + (1 - LIGHTEN) for ch in matplotlib.colors.to_rgb(c))


split_long = pd.DataFrame(
    [{"Strategie": SPLIT_LABELS[s], "Selektor": sel, "RMSE": _rmse(seed_results[sd][k])}
     for s in REC_STRATEGIES
     for sel, k in zip(SEL_LABELS, (s, f"{s}_rec"))
     for sd in seed_results])
_x_order = [SPLIT_LABELS[s] for s in REC_STRATEGIES]


def split_violin(ax):
    for s in REC_STRATEGIES:
        c = STRAT_COLORS[s]
        sns.violinplot(data=split_long[split_long["Strategie"] == SPLIT_LABELS[s]],
                       x="Strategie", y="RMSE", hue="Selektor",
                       order=_x_order, hue_order=SEL_LABELS,
                       palette={SEL_LABELS[0]: _light(c), SEL_LABELS[1]: c},
                       split=True, cut=0, density_norm="width", inner=None,
                       legend=False, ax=ax)
    grey, edge = "#8c8c8c", ts.color("box")
    handles = [matplotlib.patches.Patch(facecolor=_light(grey), edgecolor=edge),
               matplotlib.patches.Patch(facecolor=grey, edgecolor=edge)]
    ax.legend(handles, list(SEL_LABELS), frameon=False)


def half_box(ax, width=0.12):
    box = ts.color("box")
    for i, strat in enumerate(_x_order):
        for side, sel in zip((-1, 1), SEL_LABELS):
            v = split_long.query("Strategie == @strat and Selektor == @sel")["RMSE"].to_numpy()
            q1, med, q3 = np.percentile(v, [25, 50, 75])
            lo = v[v >= q1 - 1.5 * (q3 - q1)].min()
            hi = v[v <= q3 + 1.5 * (q3 - q1)].max()
            ax.add_patch(plt.Rectangle((min(i, i + side * width), q1), width, q3 - q1,
                                       facecolor=box, edgecolor=box, zorder=3))
            ax.plot([i, i + side * width], [med, med], color="w", lw=2, zorder=4)
            ax.plot([i, i], [lo, q1], color=box, lw=1.5, zorder=3)
            ax.plot([i, i], [q3, hi], color=box, lw=1.5, zorder=3)
            for y in (lo, hi):
                ax.plot([i, i + side * width / 2], [y, y], color=box, lw=1.5, zorder=3)


fig_split_box, ax = plt.subplots(figsize=(ts.fig_width(0.85), ts.fig_width(0.42)))
split_violin(ax)
half_box(ax)
ax.set_xlabel("Strategie")
ax.set_ylabel("RMSE")
ax.grid(True, axis="y", alpha=0.2)
fig_split_box.tight_layout()
if IS_FINAL:
    plot_store.save_figure(fig_split_box, "adaptation_seed_violin", adapt_cfg,
                           final=IS_FINAL, savefig_kwargs={"dpi": 300},
                           archive_kwargs={"dpi": 300})
plt.show()

## Ergebnisse speichern

Kennzahlen sind Seed-Mittel und deshalb durchgängig Dezimalzahlen, auch `n_train`, `n_detect` und `n_reject`. Mittelwert unter `strategies`, Streuung unter `strategies_std`. Das Screening wird vollständig abgelegt (Einzelwerte, p-Wert und relativer Abstand zum Führenden), damit die Gleichstandsregel im Text belegbar bleibt.

In [ ]:
from src.utils import results_export as rx

_keys = [(k, k, "num") for k in _METRIC_COLS]

doc = (rx.ResultDoc()
   .integer("n_samples", N)
   .integer("step_seconds", STEP_S)
   .set("samples_per_hour", SAMPLES_PER_H)
   .set("stride", int(STRIDE))
   .set("reset_indices", reset_pos)
   .integer("n_seeds", len(seed_results))
   .set("detectors_screened", list(DETECTORS))
   .set("informed_detector", BEST_DETECTOR)
   .set("screening", {d: {"rmse": float(v["rmse"]),
                          "rmse_std": float(v["rmse_std"]),
                          "rmse_seeds": [float(x) for x in v["rmse_seeds"]],
                          "p_vs_lead": float(v["p_vs_lead"]),
                          "rel_vs_lead": float(v["rel_vs_lead"]),
                          "n_train": float(v["n_train"])}
                      for d, v in screening.items()})
   .set("config", ADAPT_CFG)
   .set("tuning", {
       "step_tune_seconds": int(STEP_TUNE_S),
       "n_trials": N_TRIALS,
       "rec_trials": REC_TRIALS,
       "int_upper": INT_UPPER,
       "top_k": TOP_K,
       "crosscheck_won": CC_WON,
       "sampler": da.TPE_SETTINGS,
       "weights": list(TUNE_WEIGHTS),
       "best_params": best_adapt_params,
   })
   .set("cooldown", {"tuned": True, "x_range": [0.0, 60.0], "t_stat_points": float(T_stat)})
   .set("applied_config", applied_cfg)
   .set("seed_eval", {
       "seeds": [int(s) for s in SEEDS],
       "rmse": {s: {"mean": float(seed_rmse.loc[s, "mean"]),
                    "std": float(seed_rmse.loc[s, "std"])} for s in STRATEGIES}})
   .set("selector", {
       "table": {r: {c: float(v) for c, v in row.items()} for r, row in rec_table.iterrows()},
       "rmse_seeds": {r: [_rmse(seed_results[sd][r]) for sd in seed_results]
                      for r in rec_table.index}})
   .stats(stats_mean, _keys, into="strategies", rows=list(STRATEGIES))
   .stats(stats_std, _keys, into="strategies_std", rows=list(STRATEGIES)))

doc.save(res_store, "adaptation", adapt_cfg, final=IS_FINAL,
         parents={"model": scored_error_cfg.id})

### LaTeX-Makros

Detektornamen für den Fließtext: der Schwellwert-Detektor ohne eigenes Akronym, die übrigen über `\ac{...}`. Seed-Mittel von Nachtrainings und Detektionen sind nicht ganzzahlig und erhalten eine Nachkommastelle. Zu jeder Kennzahl gibt es ein `...Std`-Makro für die Darstellung Mittelwert ± Streuung.

In [ ]:
from src.utils import latex_export as lx

mx = lx.MacroExport("automatisch erzeugt aus tep_adaptation.ipynb - nicht manuell editieren")

mx.comment("Parameter (Hash-relevant)")
mx.param("tepadaptTa",     STEP_S,      unit=r"\second")
mx.param("tepadaptTaTune", STEP_TUNE_S, unit=r"\second")
mx.param("tepadaptNPoints", N)
mx.param("tepadaptChunk",   CHUNK)
mx.param("tepadaptNScreenSeeds", len(SCREENING_SEEDS), spell_out=True)
mx.param("tepadaptTopK", TOP_K, spell_out=True)

mx.comment("Detektor-Screening (Gleichstandsregel)")
_DET_LABEL = {"RMSE": "Schwellwert-Detektor", "DDM": r"\ac{DDM}",
              "EDDM": r"\ac{EDDM}", "ADWIN": r"\ac{ADWIN}", "KSWIN": r"\ac{KSWIN}"}
mx.num("tepadaptTieAlpha", TIE_ALPHA, 2)
mx.num("tepadaptTieMaxRel", 100 * TIE_MAX_REL, 0)
mx.raw("tepadaptTieOrder", ", ".join(_DET_LABEL[d] for d in TIE_ORDER))
mx.raw("tepadaptScreenLead", _DET_LABEL[SCREEN_LEAD])
mx.raw("tepadaptDetector",   _DET_LABEL[BEST_DETECTOR])

mx.comment("Ergebnisse je Strategie (Mittelwert und Std ueber alle Seeds)")
mx.integer("tepadaptNSeeds", len(seed_results))
_name_map = {"baseline": "Baseline", "blind": "Blind", "informed": "Informed",
             "combined": "Combined", "blind_rec": "BlindRec",
             "informed_rec": "InformedRec", "combined_rec": "CombinedRec"}
_mkeys = [("rmse", "Rmse", 3), ("bias", "Bias", 3), ("sigma", "Sigma", 3),
          ("corr_abs", "CorrAbs", 2), ("rmse_improvement", "Improve", 3),
          ("n_train", "NTrain", 1), ("n_detect", "NDetect", 1), ("n_recall", "NRecall", 1)]
for k in STRATEGIES:
    m = _name_map[k]
    for col, suf, nd in _mkeys:
        mx.num(f"tepadapt{m}{suf}",    stats_mean.loc[k, col], nd)
        mx.num(f"tepadapt{m}{suf}Std", stats_std.loc[k, col],  nd)
    mx.num(f"tepadapt{m}RmseMin", seed_rmse.loc[k, list(seed_results)].min(), 3)

mx.comment("Selektor: Differenz mit gegen ohne, gepaart ueber die Seeds")
for _r in rec_table.index:
    m = _name_map[_r]
    mx.num(f"tepadapt{m}Delta",    rec_table.loc[_r, "delta"], 4)
    mx.num(f"tepadapt{m}DeltaStd", rec_table.loc[_r, "delta_std"], 4)
    mx.num(f"tepadapt{m}DeltaRel", 100 * rec_table.loc[_r, "delta_rel"], 1)
    mx.num(f"tepadapt{m}PValue",   rec_table.loc[_r, "p_value"], 3)
    mx.integer(f"tepadapt{m}NBetter", rec_table.loc[_r, "n_better"])

mx.save(res_store, "adaptation", adapt_cfg, final=IS_FINAL)

In [ ]:
status.finish("\n".join(f"{s}: RMSE {stats_mean.loc[s, 'rmse']:.3f} +/- {stats_std.loc[s, 'rmse']:.3f}"
                         for s in STRATEGIES))